In [1]:
SEPARATORS = ["\n\n", "\n", "。", "!", "?", ";", ". ", "! ", "? ", "; ", " ", ""]

EMBED_MODEL = "intfloat/multilingual-e5-base"

# RERANKING_MODEL = "BAAI/bge-reranker-v2-m3"

CHUNK_SIZE_CHARS = 800
CHUNK_OVERLAP_CHARS = 200

COLLECTION = "test_bulletins"

In [2]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

c:\Users\David Gunawan Wisno\Documents\project\final-project\oir-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

model = SentenceTransformer(EMBED_MODEL)

VECTOR_SIZE = model.get_embedding_dimension()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE_CHARS,
    chunk_overlap=CHUNK_OVERLAP_CHARS,
    separators=SEPARATORS,
    keep_separator="end",  # sentence-ending punctuation stays attached
)

print("Model loaded:", EMBED_MODEL)
print("Vertor size:", VECTOR_SIZE)
print("Qdrant collections:", [c.name for c in client.get_collections().collections])

C:\Users\David Gunawan Wisno\AppData\Local\Temp\ipykernel_17104\1165165812.py:1: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1790.40it/s]


Model loaded: intfloat/multilingual-e5-base
Vertor size: 768
Qdrant collections: ['test_bulletins', 'test_collection_e5_base']


In [4]:
from qdrant_client.models import Distance, VectorParams

# if not client.collection_exists(COLLECTION):
#     client.create_collection(
#         collection_name=COLLECTION,
#         vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
#     )
#     print("Created collection:", COLLECTION)
# else:
#     print("Collection already exists:", COLLECTION)

client.delete_collection(COLLECTION)

client.create_collection(collection_name=COLLECTION, vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE))
print("Fresh collection:", COLLECTION)

Fresh collection: test_bulletins


#### Add Prefix

In [5]:
def embed_passages(texts):
    return model.encode([f"passage: {t}" for t in texts], normalize_embeddings=True).tolist()

def embed_query(text):
    return model.encode(f"query: {text}", normalize_embeddings=True).tolist()

In [6]:
import uuid
from qdrant_client.models import PointStruct

In [7]:
# 2. List all the files you want to upload
documents_to_upload = [
    "mock-data/tunghai_academic_guide.txt",
    "mock-data/tunghai_life_guide.txt"
]

points = []

In [8]:
# 3. Loop through each file
for file_path in documents_to_upload:
    print(f"Processing {file_path}...")
    
    # Extract just the file name (e.g., 'tunghai_academic_guide.txt') to store in Qdrant
    name = os.path.basename(file_path)
    
    # Open the file using the full path
    with open(file_path, encoding="utf-8") as file:
        chunks = splitter.split_text(file.read())

    vectors = embed_passages(chunks)
    points = [
        PointStruct(
            id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{name}:{i}")),
            vector=vec,
            payload = {"source_file": name, "content": text}
        )
        for i, (text, vec) in enumerate(zip(chunks, vectors))
    ]
    client.upsert(collection_name=COLLECTION, points=points)
    print(f"{name}: {len(points)} chunks")

print("Total points:", client.count(COLLECTION).count)

Processing mock-data/tunghai_academic_guide.txt...
tunghai_academic_guide.txt: 2 chunks
Processing mock-data/tunghai_life_guide.txt...
tunghai_life_guide.txt: 2 chunks
Total points: 4


In [9]:
test_new_file = "mock-data/mock-arc.txt"

with open(test_new_file, "r", encoding="utf-8") as file:
    new_document_text = file.read()

In [ ]:
search_vector = embed_query(new_document_text)

In [11]:
search_results = client.query_points(
    collection_name=COLLECTION,
    query=search_vector,
    limit=2
)

In [12]:
print(f"--- Search Results for: {os.path.basename(test_new_file)} ---")

for idx, hit in enumerate(search_results.points):
    print(f"\nMatch #{idx + 1}")
    print(f"Similarity Score: {hit.score:.4f}")
    print(f"Source File: {hit.payload['source_file']}")
    print(f"Matched Content:\n{hit.payload['content']}")
    print("-" * 50)

--- Search Results for: mock-arc.txt ---

Match #1
Similarity Score: 0.9203
Source File: tunghai_academic_guide.txt
Matched Content:
Tunghai University — Academic & Immigration Guide

1. ALIEN RESIDENT CERTIFICATE (ARC) REGISTRATION
All international students staying in Taiwan for more than six months must apply for an Alien Resident Certificate within 15 days of arrival. Applications are submitted to the National Immigration Agency office in Taichung City. Processing takes 10 working days, and the fee is NT$1,000.

2. COURSE CREDIT TRANSFER POLICY
Exchange and degree-seeking students may apply to transfer credits earned at their home institution. Applications must be submitted to the Office of Academic Affairs within the first four weeks of the semester. A maximum of one third of the total credits required for the degree may be transferred.
--------------------------------------------------

Match #2
Similarity Score: 0.8728
Source File: tunghai_life_guide.txt
Matched Content:
3. HEAL

In [13]:
# Get the best match from the database
best_match = search_results.points[0]
best_score = best_match.score

print(f"Top Vector Score: {best_score:.2f}")

Top Vector Score: 0.92


In [14]:
from openai import OpenAI

clientLlm = OpenAI(base_url="http://localhost:11434/v1", api_key="unused")

resp = clientLlm.chat.completions.create(
    model="qwen3:4b",
    messages=[{"role": "user", "content": "Reply with only: OK"}],
    temperature=0,
)
print(resp.choices[0].message.content)

OK


In [ ]:
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal, Optional

class RoutingDecision(BaseModel):
    action: Literal["UPDATE", "NEW_BULLETIN"]
    target_file: Optional[str]
    reasoning: str
    proposed_content: str

clientLlm = OpenAI(base_url="http://localhost:11434/v1", api_key="unused")

resp = clientLlm.chat.completions.create(
    model="qwen3:4b",
    messages=[
        {"role": "system", "content": "You route incoming OIR documents. Output JSON only, matching the schema exactly."},
        {"role": "user", "content": (
            "New document:\nARC renewal deadline moved to Nov 30.\n\n"
            "Existing chunk (source: arc_guide.md, score 0.81):\n"
            "Students must renew their ARC before Oct 31."
        )},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "routing_decision",
            "schema": RoutingDecision.model_json_schema(),
        },
    },
    temperature=0,
)

raw = resp.choices[0].message.content
print("RAW:", raw)
print("PARSED:", RoutingDecision.model_validate_json(raw))

RAW: {
  "action": "UPDATE",
  "target_file": "arc_guide.md",
  "reasoning": "New document indicates ARC renewal deadline moved to Nov 30, conflicting with existing chunk's deadline of Oct 31. Existing chunk has high confidence (0.81) for the original deadline, so update is required.",
  "proposed_content": "Students must renew their ARC before Nov 30."
}
PARSED: action='UPDATE' target_file='arc_guide.md' reasoning="New document indicates ARC renewal deadline moved to Nov 30, conflicting with existing chunk's deadline of Oct 31. Existing chunk has high confidence (0.81) for the original deadline, so update is required." proposed_content='Students must renew their ARC before Nov 30.'


: 